In [ ]:
!pip install -q qiskit qiskit-machine-learning kagglehub torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from kagglehub import dataset_download
from sklearn.metrics import classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit import QuantumCircuit

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
FEATURE_DIM = 8          # ⚠️ keep ≤ 8 for feasibility
EPOCHS = 1
LR = 1e-4

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
import os
import torch
from kagglehub import dataset_download
from torchvision import datasets, transforms
from torch.utils.data import ConcatDataset, random_split, DataLoader
from collections import Counter

# ----------------------------------
# Download dataset
# ----------------------------------
dataset_root = dataset_download("ismailpromus/skin-diseases-image-dataset")

# After download this should point to the base image folder
# (adjust if KaggleHub places it in a subfolder)
data_root = os.path.join(dataset_root, "IMG_CLASSES")

train_path = data_root
print("Dataset root:", dataset_root)
print("Train path:", os.listdir(train_path))

Using Colab cache for faster access to the 'skin-diseases-image-dataset' dataset.
Dataset root: /kaggle/input/skin-diseases-image-dataset
Train path: ['1. Eczema 1677', '10. Warts Molluscum and other Viral Infections - 2103', '4. Basal Cell Carcinoma (BCC) 3323', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '5. Melanocytic Nevi (NV) - 7970', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k', '3. Atopic Dermatitis - 1.25k', '6. Benign Keratosis-like Lesions (BKL) 2624', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '2. Melanoma 15.75k']


In [ ]:
# ----------------------------------
# Transforms
# ----------------------------------
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ----------------------------------
# Load ImageFolder datasets
# ----------------------------------
train_data = datasets.ImageFolder(train_path, transform=transform)

# ----------------------------------
# Combine all & split
# ----------------------------------
full_dataset = train_data

train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------------------
# Class names (multi-class)
# ----------------------------------
class_names = train_data.classes
print("Classes:", class_names)
# Example: ['Atopic Dermatitis','BCC','Benign Keratosis-like Lesions',...]

# ----------------------------------
# Count per class
# ----------------------------------
def count_classes(subset, class_names):
    counter = Counter()
    for _, label in subset:
        counter[class_names[label]] += 1
    return dict(counter)

train_counts = count_classes(train_ds, class_names)
val_counts   = count_classes(val_ds, class_names)

print("\nTraining dataset counts:")
for cls in class_names:
    print(f"  {cls}: {train_counts.get(cls,0)}")

print("\nValidation dataset counts:")
for cls in class_names:
    print(f"  {cls}: {val_counts.get(cls,0)}")

Classes: ['1. Eczema 1677', '10. Warts Molluscum and other Viral Infections - 2103', '2. Melanoma 15.75k', '3. Atopic Dermatitis - 1.25k', '4. Basal Cell Carcinoma (BCC) 3323', '5. Melanocytic Nevi (NV) - 7970', '6. Benign Keratosis-like Lesions (BKL) 2624', '7. Psoriasis pictures Lichen Planus and related diseases - 2k', '8. Seborrheic Keratoses and other Benign Tumors - 1.8k', '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']

Training dataset counts:
  1. Eczema 1677: 1354
  10. Warts Molluscum and other Viral Infections - 2103: 1650
  2. Melanoma 15.75k: 2486
  3. Atopic Dermatitis - 1.25k: 995
  4. Basal Cell Carcinoma (BCC) 3323: 2685
  5. Melanocytic Nevi (NV) - 7970: 6427
  6. Benign Keratosis-like Lesions (BKL) 2624: 1658
  7. Psoriasis pictures Lichen Planus and related diseases - 2k: 1638
  8. Seborrheic Keratoses and other Benign Tumors - 1.8k: 1448
  9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k: 1381

Validation dataset counts:
  1. Ec

In [ ]:
feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)

qc = QuantumCircuit(FEATURE_DIM)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

/tmp/ipython-input-86803684.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
/tmp/ipython-input-86803684.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)


In [ ]:
qc.decompose().draw()

┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                  »
«q_0: ────────────────────────────────■─────────■──»
«                                     │         │  »
«q_1: ────────────────────────────────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐┌─┴─┐  │  »
«q_2: ┤ P(2*(π - x[0])*(π - x[2])) ├┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘└───┘└───┘┌─┴─┐»
«q_3: ────────────────────────────────────────┤ X ├»
«                                             └───┘»
«q_4: ─────────────────────────────────────────────»
«                                                  »
«q_5: ─────────────────────────────────────────────»
«                                                  »
«q_6: ─────────────────────────────────────────────»
«                                                  »
«q_7: ─────────────────────────────────────────────»
«                                                  »
«                                                       »
«q_0: ─────────────────────────────────────■─────────■──»
«                                          │         │  »
«q_1: ────────────────────────────────■────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │    │    │  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──┼────┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐┌─┴─┐  │  »
«q_3: ┤ P(2*(π - x[0])*(π - x[3])) ├─────┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘     └───┘└───┘┌─┴─┐»
«q_4: ─────────────────────────────────────────────┤ X ├»
«                                                  └───┘»
«q_5: ──────────────────────────────────────────────────»
«                                                       »
«q_6: ──────────────────────────────────────────────────»
«                                                       »
«q_7: ──────────────────────────────────────────────────»
«                                                       »
«                                                            »
«q_0: ─────────────────────────────────────■──────────────■──»
«                                          │              │  »
«q_1: ────────────────────────────────■────┼─────────■────┼──»
«                                     │    │         │    │  »
«q_2: ────────────────────────────────┼────┼────■────┼────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │  ┌─┴─┐  │    │  »
«q_3: ┤ P(2*(π - x[1])*(π - x[3])) ├┤ X ├──┼──┤ X ├──┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐└───┘┌─┴─┐  │  »
«q_4: ┤ P(2*(π - x[0])*(π - x[4])) ├─────┤ X ├─────┤ X ├──┼──»
«     └────────────────────────────┘     └───┘     └───┘┌─┴─┐»
«q_5: ──────────────────────────────────────────────────┤ X ├»
«                                                       └───┘»
«q_6: ───────────────────────────────────────────────────────»
«                                             

In [ ]:
class HybridEndToEnd(nn.Module):
    def __init__(self, qnn, feature_dim):
        super().__init__()

        # ---------------------------
        # Classical CNN (VGG19)
        # ---------------------------
        self.cnn = models.vgg19(pretrained=True)

        # Remove VGG classifier
        self.cnn.classifier = nn.Identity()

        # Feature bottleneck → QNN
        self.reduce = nn.Sequential(
            nn.Linear(25088, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim),
            nn.Tanh()
        )

        # ---------------------------
        # Quantum Layer
        # ---------------------------
        self.qnn = TorchConnector(qnn)

        # Final classifier
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        x = self.cnn(x)
        x = self.reduce(x)
        x = self.qnn(x)
        return torch.sigmoid(self.fc(x))

In [ ]:
model = HybridEndToEnd(qnn, FEATURE_DIM).to(device)

# Freeze VGG weights for stability
for param in model.cnn.parameters():
    param.requires_grad = False

# Leave bottleneck + QNN trainable

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 236MB/s]


In [ ]:
criterion = nn.BCELoss()  #this is binary cross-entropy loss
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    TP = FP = TN = FN = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs >= 0.5).float()

        TP += ((preds == 1) & (labels == 1)).sum().item()
        TN += ((preds == 0) & (labels == 0)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Acc: {accuracy:.2f}% | "
        f"Prec: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )

In [ ]:
torch.save(model,'hybrid_vgg_qnn.pt')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        preds = model(images)
        y_pred.extend(preds.cpu().numpy().round().astype(int).flatten())
        y_true.extend(labels.cpu().numpy().astype(int))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# Unfreeze last convolution block
for param in model.cnn.features[-5:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
print("Last VGG block unfrozen")

In [ ]:
criterion = nn.BCELoss()  #this is binary cross-entropy loss
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    TP = FP = TN = FN = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs >= 0.5).float()

        TP += ((preds == 1) & (labels == 1)).sum().item()
        TN += ((preds == 0) & (labels == 0)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Acc: {accuracy:.2f}% | "
        f"Prec: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )

In [ ]:
torch.save(model,'hybrid_vgg_qnn_all_unfrozen.pt')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        preds = model(images)
        y_pred.extend(preds.cpu().numpy().round().astype(int).flatten())
        y_true.extend(labels.cpu().numpy().astype(int))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
